# 00 — Dataset Construction & Disease Label Extraction

**Project:** Retrieval-Augmented Medical Vision-Language Learning for Explainable
Ultrasound Disease Classification and Report Generation

This notebook covers **Phase 1** of the pipeline. It turns the raw
`Reports.xlsx` + image folder into a single clean, versioned master table
that every downstream notebook consumes.

What it produces in `/kaggle/working/artifacts/`:

| File | Contents |
|---|---|
| `master.csv` | one row per study: image path, raw report, cleaned report, sections, labels |
| `labels_multihot.csv` | multi-hot disease matrix |
| `ontology.json` | the label rule-set actually used (put this in your paper appendix) |
| `label_audit.csv` | impression text next to assigned labels — **you must eyeball this** |

> **Read this before running.** Label extraction here is rule-based over the
> Impression/Comment section. It is a *first pass*, not ground truth. Phase 1 of
> a defensible paper is: run this, open `label_audit.csv`, correct what is wrong,
> and save the corrected file back. Everything downstream will pick up the
> corrections automatically.

In [1]:
import os
import sys
import subprocess

def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

try:
    import openpyxl  # noqa
except ImportError:
    pipq("openpyxl")

import json
import re
import shutil
import collections
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 160)
print("pandas", pd.__version__)

pandas 2.3.3


## 1. Paths

Upload two things to Kaggle as **one dataset** (call it e.g. `ultrasound-fydp`):

```
ultrasound-fydp/
  Reports.xlsx
  images/
    R1.jpg
    R2.jpg
    ...
    R259.jpg
```

The cell below auto-discovers them under `/kaggle/input`, so you do not have to
hard-code the dataset slug. If you run locally, set `LOCAL_ROOT`.

In [2]:
LOCAL_ROOT = None  # e.g. "/home/me/ultrasound"  — leave None on Kaggle

def discover_root():
    if LOCAL_ROOT:
        return Path(LOCAL_ROOT)
    base = Path("/kaggle/input/datasets/ashrafbin/ultrasound-data/Ultrasound Data")
    if not base.exists():
        raise FileNotFoundError("No /kaggle/input. Set LOCAL_ROOT.")
    for ds in sorted(base.iterdir()):
        if list(ds.rglob("*.xlsx")) or list(ds.rglob("*.csv")):
            return ds
    return sorted(base.iterdir())[0]

ROOT = discover_root()
print("Dataset root:", ROOT)

xlsx_hits = sorted(ROOT.rglob("*.xlsx")) + sorted(ROOT.rglob("*.xls"))
csv_hits = sorted(ROOT.rglob("*.csv"))
print("Spreadsheets found:", [p.name for p in xlsx_hits + csv_hits])

REPORT_FILE = xlsx_hits[0] if xlsx_hits else csv_hits[0]

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
all_images = [p for p in ROOT.rglob("*") if p.suffix.lower() in IMG_EXT]
print(f"Images found: {len(all_images)}")
if all_images:
    print("Example:", all_images[0])

ART = Path("/kaggle/working/artifacts")
ART.mkdir(parents=True, exist_ok=True)
print("Artifacts ->", ART)

Dataset root: /kaggle/input/datasets/ashrafbin/ultrasound-data/Ultrasound Data/Ultrasound Data
Spreadsheets found: ['Reports.xlsx']
Images found: 259
Example: /kaggle/input/datasets/ashrafbin/ultrasound-data/Ultrasound Data/Ultrasound Data/Images/R139.jpg
Artifacts -> /kaggle/working/artifacts


## 2. Load reports

In [3]:
if REPORT_FILE.suffix.lower().startswith(".xls"):
    raw = pd.read_excel(REPORT_FILE)
else:
    raw = pd.read_csv(REPORT_FILE)

raw.columns = [c.strip() for c in raw.columns]
print(raw.shape, list(raw.columns))

ID_COL = next((c for c in raw.columns if "image" in c.lower() or "id" in c.lower()), raw.columns[0])
TXT_COL = next((c for c in raw.columns if c != ID_COL and raw[c].astype(str).str.len().mean() > 100), raw.columns[1])
print(f"id column = {ID_COL!r}   text column = {TXT_COL!r}")

df = raw[[ID_COL, TXT_COL]].copy()
df.columns = ["study_id", "report_raw"]
df["study_id"] = df["study_id"].astype(str).str.strip()
df["report_raw"] = df["report_raw"].astype(str)
df = df[df["report_raw"].str.len() > 20].reset_index(drop=True)
print("Usable rows:", len(df))
df.head(3)

(259, 2) ['Ultrasound_Image', 'Description']
id column = 'Ultrasound_Image'   text column = 'Description'
Usable rows: 259


,study_id,report_raw
0,R1,ULTRASOUND EXAMINATION OF WHOLE ABDOMEN\n\nClinical Information: * Abdominal pain\n\nLiver: Normal in size with homogeneous echotexture. No focal or diffuse...
1,R2,"ULTRASONOGRAPHY OF WHOLE ABDOMEN\n\nLiver : Normal in size. Hepatic parenchymal echotexture is homogenous.\nNo focal solid, cystic or any mixed echogenic le..."
2,R3,TRANSVAGINAL SONOGRAPHY\n\nUterus:\n\nShape: Normal Inverted pear shape.\n\nPosition: Anteverted and Antiflexed.\n\nSize: Normal in size (5.36 x 3.17 x 2.47...


## 3. Attach images

Study `R17` is matched to `R17.jpg` (case-insensitive, any supported extension).
Studies with no image are kept but flagged — the text-only baseline can still
use them, and reporting how many you dropped is a sentence your Dataset section needs.

In [4]:
img_lookup = {}
for p in all_images:
    img_lookup.setdefault(p.stem.strip().lower(), p)

df["image_path"] = df["study_id"].str.lower().map(lambda s: str(img_lookup.get(s, "")))
df["has_image"] = df["image_path"] != ""

print(f"matched   : {df['has_image'].sum()}")
print(f"unmatched : {(~df['has_image']).sum()}")
if (~df["has_image"]).any():
    print("Missing image for:", df.loc[~df["has_image"], "study_id"].tolist()[:20])

orphan = set(img_lookup) - set(df["study_id"].str.lower())
print(f"images with no report: {len(orphan)}", sorted(orphan)[:10])

matched   : 259
unmatched : 0
images with no report: 0 []


## 4. Text normalisation and boilerplate removal

Ultrasound reports from clinical PACS carry a fixed legal footer that appears on
a subset of studies. It carries zero diagnostic signal but is long enough to
dominate BLEU/ROUGE and to skew TF-IDF. We strip it and keep the removed span so
the choice is auditable.

We deliberately do **not** lowercase, stem, or strip punctuation — measurements
(`7.9mm`), grades (`Gr-II`), and abbreviations (`CBD`, `MPD`, `PVR`) are signal.

In [5]:
BOILERPLATE = [
    r"Ultrasound is not ideal imaging technique[^.]*\.",
    r"(?:This report|It) should be interpreted in correlation[^.]*\.",
    r"Medico-?legal use of this report[^.]*\.",
    r"Please notify us for typing mistake[^.]*\.",
    r"Please bring all previous records[^.]*\.",
    r"Disclaimer\s*:\s*The conclusion is based upon[^.]*\.",
    r"The conclusion is based upon ultrasound images[^.]*\.",
    r"it'?s only a radiological impression\.?",
]

UNICODE_FIX = {
    "\u2019": "'", "\u2018": "'", "\u201c": '"', "\u201d": '"',
    "\u2013": "-", "\u2014": "-", "\u00a0": " ", "\u2026": "...",
    "\u0441": "c", "\u0435": "e", "\u043e": "o", "\u0430": "a", "\u0442": "t",
}

def normalise(text: str) -> str:
    for k, v in UNICODE_FIX.items():
        text = text.replace(k, v)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def strip_boilerplate(text: str):
    removed = []
    for pat in BOILERPLATE:
        for m in re.finditer(pat, text, flags=re.I | re.S):
            removed.append(m.group(0))
        text = re.sub(pat, " ", text, flags=re.I | re.S)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text)
    return text.strip(), removed

df["report_norm"] = df["report_raw"].map(normalise)
stripped = df["report_norm"].map(strip_boilerplate)
df["report_clean"] = [s[0] for s in stripped]
df["boilerplate_removed"] = [len(s[1]) for s in stripped]

print("studies containing boilerplate:", (df["boilerplate_removed"] > 0).sum())
print("mean chars removed:",
      (df["report_norm"].str.len() - df["report_clean"].str.len()).mean().round(1))

studies containing boilerplate: 6
mean chars removed: 8.8


## 5. Section segmentation

**This is contribution #2 in your proposal ("section-aware report encoding"), so
it needs to be done properly rather than by splitting on newlines.**

Reports follow an organ-by-organ template that varies by exam type. We segment
into named anatomical/structural sections plus the two sections that matter most
for the downstream tasks: `FINDINGS` (everything before the impression) and
`IMPRESSION` (the diagnostic conclusion).

In [6]:
SECTION_HEADS = [
    "CLINICAL INFORMATION", "CLINICAL INFO", "INDICATION",
    "LIVER", "GALL BLADDER", "GALLBLADDER", "GB", "BILIARY TREE", "BILIARY TREES",
    "CBD", "PANCREAS", "SPLEEN", "KIDNEY", "KIDNEYS", "RIGHT KIDNEY", "LEFT KIDNEY",
    "URETER", "URETERS", "ADRENAL GLANDS", "AORTA & IVC", "AORTA AND IVC",
    "URINARY BLADDER", "U.BLADDER", "U. BLADDER", "BLADDER", "PROSTATE",
    "UTERUS", "MYOMETRIUM", "ENDOMETRIUM", "CERVIX", "ADNEXA",
    "RIGHT OVARY", "LEFT OVARY", "OVARIES", "CUL-DE-SAC", "CUL-DE-SAC",
    "P.O.D", "POD", "PLACENTA", "AMNIOTIC FLUID", "FETUS", "FETAL BIOMETRICS",
    "FETAL AGE", "OTHERS", "RECTUM",
]
HEAD_ALT = "|".join(sorted((re.escape(h) for h in SECTION_HEADS), key=len, reverse=True))
HEAD_RE = re.compile(rf"(?m)^\s*({HEAD_ALT})\s*:?\s*", re.I)

IMPRESSION_RE = re.compile(
    r"\b(IMPRESSIONS?|COMMENTS?|OPINION|CONCLUSION|SUMMARY)\b\s*:?\s*", re.I
)
ADVICE_RE = re.compile(r"\b(ADV|ADVICE|ADVISED|RECOMMENDATION)\b\s*:?\s*", re.I)

def split_impression(text: str):
    """Return (findings, impression, advice)."""
    matches = list(IMPRESSION_RE.finditer(text))
    if not matches:
        return text.strip(), "", ""
    m = matches[-1]
    tail = text[m.end():]
    if len(re.sub(r"\W", "", tail)) < 3 and len(matches) > 1:
        m = matches[0]
        tail = text[m.end():]
    findings = text[: m.start()].strip()
    adv = ADVICE_RE.search(tail)
    if adv:
        advice = tail[adv.end():].strip()
        impression = tail[: adv.start()].strip()
    else:
        advice, impression = "", tail.strip()
    return findings, re.sub(r"\s+", " ", impression).strip(), re.sub(r"\s+", " ", advice).strip()

def split_sections(findings: str) -> dict:
    out, marks = {}, list(HEAD_RE.finditer(findings))
    if not marks:
        return {"BODY": re.sub(r"\s+", " ", findings).strip()}
    if marks[0].start() > 0:
        head = findings[: marks[0].start()].strip()
        if head:
            out["HEADER"] = re.sub(r"\s+", " ", head)
    for i, m in enumerate(marks):
        end = marks[i + 1].start() if i + 1 < len(marks) else len(findings)
        key = m.group(1).upper().replace(".", "").strip()
        body = re.sub(r"\s+", " ", findings[m.end(): end]).strip(" .;")
        if body:
            out[key] = (out.get(key, "") + " " + body).strip()
    return out

parts = df["report_clean"].map(split_impression)
df["findings"] = [p[0] for p in parts]
df["impression"] = [p[1] for p in parts]
df["advice"] = [p[2] for p in parts]
df["sections"] = df["findings"].map(split_sections)
df["n_sections"] = df["sections"].map(len)

print("studies with an impression :", (df["impression"].str.len() > 3).sum(), "/", len(df))
print("studies with advice line   :", (df["advice"].str.len() > 3).sum())
print("mean sections per report   :", round(df["n_sections"].mean(), 2))
df[["study_id", "impression"]].head(8)

studies with an impression : 259 / 259
studies with advice line   : 38
mean sections per report   : 8.31


,study_id,impression
0,R1,Suggestive of Inhomogeneous uterus.
1,R2,Right adnexal chocolate cyst.
2,R3,"Normal size uterus, endometrial thickness is 7.9mm. Multiple nabothian cysts."
3,R4,Suggestive of early pregnancy failure (Missed abortion).
4,R5,Suggestive of Enlarged right ovary with endometrioma.
5,R6,Normal USG findings.
6,R7,Normal study at ultrasound.
7,R8,1. An early intrauterine pregnancy of about 05+ weeks. 2. Mild pelvic collection.


## 6. Exam-type detection

The corpus mixes five protocols. This is a clean, almost noise-free auxiliary
label — it trains reliably even at n=259, and it is what makes the multi-task
story in Phase 8 work when the disease head is starved for positives.
It also lets you report *stratified* results, which reviewers ask for.

In [7]:
def exam_type(text: str) -> str:
    h = text[:200].upper()
    if "PREGNANCY PROFILE" in h or "OBSTETRIC" in h:
        return "Pregnancy profile"
    if "TRANSVAGINAL" in h or "TVS" in h:
        return "TVS"
    if "WHOLE ABDOMEN" in h:
        return "Whole abdomen"
    if "LOWER ABDOMEN" in h or "KUB" in h:
        return "Lower abdomen"
    if "BREAST" in h:
        return "Breast"
    if "THYROID" in h or "NECK" in h:
        return "Thyroid/Neck"
    if "SCROTUM" in h or "TESTIS" in h:
        return "Scrotal"
    return "Other"

df["exam_type"] = df["report_clean"].map(exam_type)
print(df["exam_type"].value_counts())

exam_type
Whole abdomen        116
Lower abdomen         56
TVS                   32
Pregnancy profile     32
Other                 14
Breast                 6
Scrotal                2
Thyroid/Neck           1
Name: count, dtype: int64


## 7. Disease label ontology

Each label is a set of regex patterns matched against the **impression only**.
Matching on the full report would fire on negations ("No evidence of calculus"),
which is the classic failure mode for rule-based labelling of radiology text.

`exclude` patterns handle the residual negations inside impressions
(e.g. "No abdominal lymphadenopathy, ascites or pleural effusion is seen").

The `group` field lets downstream notebooks collapse rare labels into an
organ-system taxonomy when the fine-grained version has too few positives.

In [8]:
ONTOLOGY = {
    "Normal study": {
        "group": "Normal",
        "include": [r"normal (?:usg |us |sonographic )?(?:study|findings|scan)",
                    r"normal study at ultrasound", r"nothing abnormal",
                    r"no (?:significant )?abnormalit", r"normal size uterus",
                    r"unremarkable", r"normal sonographic", r"otherwise normal"],
        "exclude": [],
    },
    "Fatty liver": {
        "group": "Hepatobiliary",
        "include": [r"fatty change", r"fatty liver", r"hepatic steatosis", r"steatosis"],
        "exclude": [],
    },
    "Hepatomegaly": {
        "group": "Hepatobiliary",
        "include": [r"hepatomegaly", r"enlarged liver", r"liver is enlarged"],
        "exclude": [],
    },
    "Hepatic focal lesion": {
        "group": "Hepatobiliary",
        "include": [r"ha?emangioma", r"hepatic metastas", r"liver sol",
                    r"hepatic (?:cyst|abscess)", r"focal lesion in .{0,20}liver"],
        "exclude": [],
    },
    "Hepatitis": {
        "group": "Hepatobiliary",
        "include": [r"hepatitis"],
        "exclude": [],
    },
    "Cholelithiasis": {
        "group": "Hepatobiliary",
        "include": [r"cholelithias", r"gall ?bladder (?:stone|calculus|calculi)",
                    r"\bgb (?:stone|calculus|calculi)"],
        "exclude": [],
    },
    "Gallbladder abnormality": {
        "group": "Hepatobiliary",
        "include": [r"gb polyp", r"polyp.{0,25}gall", r"thickened gb wall",
                    r"cholecystitis", r"gb sludge", r"biliary sludge",
                    r"cholecystectomy", r"gb wall thickening"],
        "exclude": [],
    },
    "Cystitis": {
        "group": "Urinary",
        "include": [r"cystitis"],
        "exclude": [],
    },
    "Significant PVR": {
        "group": "Urinary",
        "include": [r"\bpvr\b", r"post ?void residual", r"residual urine"],
        "exclude": [],
    },
    "Urinary tract infection": {
        "group": "Urinary",
        "include": [r"urinary tract infection", r"\buti\b"],
        "exclude": [],
    },
    "Renal cyst": {
        "group": "Urinary",
        "include": [r"cortical cyst", r"renal cyst", r"kidney cyst", r"polycystic kidney"],
        "exclude": [],
    },
    "Nephrolithiasis": {
        "group": "Urinary",
        "include": [r"nephrolithias", r"renal (?:stone|calculus|calculi)",
                    r"urolithias", r"ureteric calculus", r"vesical calculus"],
        "exclude": [],
    },
    "Hydronephrosis": {
        "group": "Urinary",
        "include": [r"hydronephros", r"hydroureter", r"pelvi-?calyceal.{0,20}dilat"],
        "exclude": [],
    },
    "Enlarged prostate": {
        "group": "Male pelvis",
        "include": [r"enlarged prostate", r"prostatomegaly", r"\bbph\b",
                    r"prostatic enlargement", r"prostate.{0,20}enlarged"],
        "exclude": [],
    },
    "Scrotal abnormality": {
        "group": "Male pelvis",
        "include": [r"varicocele", r"hydrocele", r"epididym", r"swollen .{0,10}testis",
                    r"gynaecomastia", r"gynecomastia"],
        "exclude": [],
    },
    "Uterine fibroid": {
        "group": "Uterine",
        "include": [r"fibroid", r"myoma(?!e)", r"leiomyoma"],
        "exclude": [],
    },
    "Adenomyosis": {
        "group": "Uterine",
        "include": [r"adenomyos"],
        "exclude": [],
    },
    "Bulky uterus": {
        "group": "Uterine",
        "include": [r"bulky uterus", r"enlarged uterus", r"uterus is enlarged",
                    r"bulky .{0,10}uterus"],
        "exclude": [r"gravid"],
    },
    "Inhomogeneous uterus": {
        "group": "Uterine",
        "include": [r"inhomogeneous uterus", r"heterogeneous.{0,15}uterus",
                    r"inhomogeneous.{0,15}myometri"],
        "exclude": [],
    },
    "Nabothian cyst": {
        "group": "Uterine",
        "include": [r"nabothian"],
        "exclude": [],
    },
    "Thickened endometrium": {
        "group": "Uterine",
        "include": [r"thickened endometrium", r"endometrial thickening",
                    r"endometrial collection", r"endometrium is thickened"],
        "exclude": [],
    },
    "Ovarian cyst": {
        "group": "Ovarian",
        "include": [r"ovarian cyst", r"chocolate cyst", r"endometrioma",
                    r"adnexal (?:complex )?cyst", r"cyst.{0,25}ovary",
                    r"complex cyst", r"multi-?cystic ovar"],
        "exclude": [],
    },
    "Polycystic ovaries": {
        "group": "Ovarian",
        "include": [r"polycystic ovar", r"\bpcos?\b"],
        "exclude": [],
    },
    "Prominent ovaries": {
        "group": "Ovarian",
        "include": [r"prominent ovar", r"enlarged ovar", r"bulky ovar",
                    r"dominant (?:right |left )?ovary"],
        "exclude": [],
    },
    "Normal pregnancy": {
        "group": "Obstetric",
        "include": [r"(?:alive|live)\b.{0,25}(?:intrauterine|\biu\b)",
                    r"intrauterine pregnancy", r"\biu pregnancy\b", r"single alive",
                    r"single live", r"weeks .{0,12}days of gestation",
                    r"estimated fetal weight", r"\bafi\b", r"cephalic"],
        "exclude": [r"failure", r"missed abortion", r"no cardiac"],
    },
    "Early pregnancy failure": {
        "group": "Obstetric",
        "include": [r"missed abortion", r"pregnancy failure", r"blighted ovum",
                    r"anembryonic", r"fetal demise", r"silent miscarriage",
                    r"no cardiac activity", r"incomplete abortion",
                    r"fetal heart.{0,20}(?:absent|not seen)"],
        "exclude": [],
    },
    "Retained products of conception": {
        "group": "Obstetric",
        "include": [r"retained (?:debris|product)", r"\brpoc\b"],
        "exclude": [],
    },
    "Pelvic collection": {
        "group": "Pelvic",
        "include": [r"pelvic collection", r"pelvic fluid", r"fluid.{0,25}cul-?de-?sac"],
        "exclude": [r"free from any fluid"],
    },
    "Pelvic inflammatory disease": {
        "group": "Pelvic",
        "include": [r"\bpid\b", r"pelvic inflammatory"],
        "exclude": [],
    },
    "Ascites": {
        "group": "Other",
        "include": [r"\bascites\b"],
        "exclude": [r"no (?:definite )?(?:abdominal )?lymphadenopathy,? ascites",
                    r"no ascites", r"ascites[^.]{0,20}(?:not|no) seen",
                    r"ascites or pleural effusion is seen"],
    },
    "Splenomegaly": {
        "group": "Other",
        "include": [r"splenomegaly", r"enlarged spleen"],
        "exclude": [],
    },
    "Thyroid nodule": {
        "group": "Other",
        "include": [r"thyroid"],
        "exclude": [],
    },
    "Breast lesion": {
        "group": "Other",
        "include": [r"fibrocystic", r"fibroadenoma", r"breast"],
        "exclude": [],
    },
    "Follicular study": {
        "group": "Ovarian",
        "include": [r"follicle", r"follicular"],
        "exclude": [],
    },
}

with open(ART / "ontology.json", "w") as f:
    json.dump(ONTOLOGY, f, indent=2)
print(f"{len(ONTOLOGY)} label definitions saved")

34 label definitions saved


In [9]:
PATHOLOGY_LABELS = [k for k in ONTOLOGY if k != "Normal study"]

def assign_labels(impression: str):
    t = impression.lower()
    hits = []
    for name in PATHOLOGY_LABELS:
        spec = ONTOLOGY[name]
        if any(re.search(p, t) for p in spec["include"]):
            if not any(re.search(p, t) for p in spec.get("exclude", [])):
                hits.append(name)
    if not hits:
        nspec = ONTOLOGY["Normal study"]
        if any(re.search(p, t) for p in nspec["include"]):
            hits = ["Normal study"]
    return hits or ["Unlabelled"]

df["labels"] = df["impression"].map(assign_labels)
df["n_labels"] = df["labels"].map(len)

counts = collections.Counter(l for ls in df["labels"] for l in ls)
label_freq = (pd.DataFrame(counts.most_common(), columns=["label", "count"])
                .assign(pct=lambda d: (100 * d["count"] / len(df)).round(1)))
label_freq["group"] = label_freq["label"].map(lambda l: ONTOLOGY.get(l, {}).get("group", "-"))
label_freq

,label,count,pct,group
0,Normal study,84,32.4,Normal
1,Normal pregnancy,42,16.2,Obstetric
2,Fatty liver,28,10.8,Hepatobiliary
3,Cystitis,21,8.1,Urinary
4,Bulky uterus,16,6.2,Uterine
5,Hepatomegaly,15,5.8,Hepatobiliary
6,Early pregnancy failure,14,5.4,Obstetric
7,Significant PVR,10,3.9,Urinary
8,Pelvic collection,7,2.7,Pelvic
9,Thickened endometrium,6,2.3,Uterine


In [10]:
print("mean labels per study:", round(df["n_labels"].mean(), 2))
print("multi-label studies  :", (df["n_labels"] > 1).sum())
print("unlabelled studies   :", (df["labels"].map(lambda x: x == ["Unlabelled"])).sum())
print()
print("Impressions the rules could not label — read these and extend ONTOLOGY:")
for t in df.loc[df["labels"].map(lambda x: x == ["Unlabelled"]), "impression"]:
    print("  •", t[:140])

mean labels per study: 1.2
multi-label studies  : 38
unlabelled studies   : 4

Impressions the rules could not label — read these and extend ONTOLOGY:
  • As per description, please.
  • As per description, please.
  • Suggestive of Bilateral axillary mild prominent fat lobules.
  • USG features are consistent with Post-inflammatory mild swelling in left tendo-achilis region.


## 8. Manual audit export

Open `label_audit.csv`, fix the `labels_corrected` column (pipe-separated, e.g.
`Fatty liver|Hepatomegaly`), re-upload it as a Kaggle dataset, and set
`CORRECTIONS` in the next cell. **Do not skip this.** A reviewer will ask how
your labels were validated, and "regex" is not an answer — "regex, then manually
reviewed by N annotators with Cohen's κ = x" is.

In [11]:
audit = df[["study_id", "exam_type", "impression"]].copy()
audit["labels_auto"] = df["labels"].map("|".join)
audit["labels_corrected"] = audit["labels_auto"]
audit["reviewer_note"] = ""
audit.to_csv(ART / "label_audit.csv", index=False)
print("Wrote", ART / "label_audit.csv")
audit.head(10)

Wrote /kaggle/working/artifacts/label_audit.csv


,study_id,exam_type,impression,labels_auto,labels_corrected,reviewer_note
0,R1,Whole abdomen,Suggestive of Inhomogeneous uterus.,Inhomogeneous uterus,Inhomogeneous uterus,
1,R2,Whole abdomen,Right adnexal chocolate cyst.,Ovarian cyst,Ovarian cyst,
2,R3,TVS,"Normal size uterus, endometrial thickness is 7.9mm. Multiple nabothian cysts.",Nabothian cyst,Nabothian cyst,
3,R4,TVS,Suggestive of early pregnancy failure (Missed abortion).,Early pregnancy failure,Early pregnancy failure,
4,R5,Whole abdomen,Suggestive of Enlarged right ovary with endometrioma.,Ovarian cyst,Ovarian cyst,
5,R6,Whole abdomen,Normal USG findings.,Normal study,Normal study,
6,R7,Whole abdomen,Normal study at ultrasound.,Normal study,Normal study,
7,R8,TVS,1. An early intrauterine pregnancy of about 05+ weeks. 2. Mild pelvic collection.,Normal pregnancy|Pelvic collection,Normal pregnancy|Pelvic collection,
8,R9,Other,Excessive bowel gas shadow present otherwise normal study.,Normal study,Normal study,
9,R10,Lower abdomen,NORMAL USG STUDY.,Normal study,Normal study,


In [12]:
CORRECTIONS = None  # e.g. "/kaggle/input/my-corrections/label_audit_reviewed.csv"

if CORRECTIONS and Path(CORRECTIONS).exists():
    fix = pd.read_csv(CORRECTIONS)
    m = dict(zip(fix["study_id"].astype(str), fix["labels_corrected"].astype(str)))
    n_changed = 0
    new_labels = []
    for sid, cur in zip(df["study_id"], df["labels"]):
        if sid in m and m[sid].strip():
            lab = [x.strip() for x in m[sid].split("|") if x.strip()]
            if lab != cur:
                n_changed += 1
            new_labels.append(lab)
        else:
            new_labels.append(cur)
    df["labels"] = new_labels
    print(f"Applied manual corrections to {n_changed} studies.")
else:
    print("No corrections file supplied — using automatic labels.")
    print("Come back to this cell after Phase 1 review.")

No corrections file supplied — using automatic labels.
Come back to this cell after Phase 1 review.


## 9. Multi-hot matrix + grouped labels

In [13]:
ALL_LABELS = sorted({l for ls in df["labels"] for l in ls if l != "Unlabelled"})
multihot = pd.DataFrame(
    [[int(l in ls) for l in ALL_LABELS] for ls in df["labels"]],
    columns=ALL_LABELS, index=df["study_id"],
)

GROUPS = sorted({ONTOLOGY.get(l, {}).get("group", "Other") for l in ALL_LABELS})
grouped = pd.DataFrame(0, index=df["study_id"], columns=GROUPS)
for l in ALL_LABELS:
    g = ONTOLOGY.get(l, {}).get("group", "Other")
    grouped[g] = np.maximum(grouped[g].values, multihot[l].values)

print("fine-grained labels:", len(ALL_LABELS))
print("grouped labels     :", len(GROUPS))
print()
print("Positives per grouped label:")
print(grouped.sum().sort_values(ascending=False))

fine-grained labels: 33
grouped labels     : 9

Positives per grouped label:
Normal           84
Obstetric        58
Hepatobiliary    39
Urinary          33
Uterine          30
Ovarian          14
Pelvic           10
Other             6
Male pelvis       5
dtype: int64


### Which labels are actually trainable?

With 5-fold CV a class needs roughly **≥10 positives** before per-fold metrics
mean anything, and ≥5 before it can even appear in every fold. Everything below
that belongs in the dataset description, not the results table. Be upfront about
this in the paper — it is a strength, not a weakness, to say so.

In [14]:
pos = multihot.sum().sort_values(ascending=False)
trainable = pos[pos >= 10].index.tolist()
marginal = pos[(pos >= 5) & (pos < 10)].index.tolist()
rare = pos[pos < 5].index.tolist()

print(f"TRAINABLE (>=10 pos, {len(trainable)}): {trainable}")
print()
print(f"MARGINAL  (5-9 pos, {len(marginal)}): {marginal}")
print()
print(f"RARE      (<5 pos, {len(rare)}): {rare}")

with open(ART / "label_tiers.json", "w") as f:
    json.dump({"trainable": trainable, "marginal": marginal, "rare": rare,
               "groups": GROUPS}, f, indent=2)

TRAINABLE (>=10 pos, 8): ['Normal study', 'Normal pregnancy', 'Fatty liver', 'Cystitis', 'Bulky uterus', 'Hepatomegaly', 'Early pregnancy failure', 'Significant PVR']

MARGINAL  (5-9 pos, 5): ['Pelvic collection', 'Renal cyst', 'Thickened endometrium', 'Cholelithiasis', 'Ovarian cyst']

RARE      (<5 pos, 20): ['Hepatic focal lesion', 'Polycystic ovaries', 'Prominent ovaries', 'Breast lesion', 'Uterine fibroid', 'Pelvic inflammatory disease', 'Inhomogeneous uterus', 'Nabothian cyst', 'Follicular study', 'Gallbladder abnormality', 'Enlarged prostate', 'Hepatitis', 'Scrotal abnormality', 'Retained products of conception', 'Adenomyosis', 'Ascites', 'Hydronephrosis', 'Splenomegaly', 'Thyroid nodule', 'Urinary tract infection']


## 10. Single-label view

Multi-label is the honest formulation, but a single-label view is needed for the
confusion matrices and McNemar tests in Phase 12. We take the most *specific*
(rarest) pathology as the primary diagnosis, falling back to `Normal study`.

In [15]:
rarity = {l: pos[l] for l in ALL_LABELS}

def primary(labels):
    path = [l for l in labels if l not in ("Normal study", "Unlabelled")]
    if not path:
        return "Normal study" if "Normal study" in labels else "Unlabelled"
    return min(path, key=lambda l: rarity.get(l, 0))

df["primary_label"] = df["labels"].map(primary)
print(df["primary_label"].value_counts())

primary_label
Normal study                       84
Normal pregnancy                   39
Early pregnancy failure            14
Bulky uterus                       12
Cystitis                           12
Fatty liver                        11
Hepatomegaly                        9
Pelvic collection                   6
Significant PVR                     6
Thickened endometrium               5
Unlabelled                          4
Ovarian cyst                        4
Renal cyst                          4
Inhomogeneous uterus                3
Pelvic inflammatory disease         3
Uterine fibroid                     3
Enlarged prostate                   3
Nabothian cyst                      3
Polycystic ovaries                  3
Hepatic focal lesion                3
Cholelithiasis                      3
Prominent ovaries                   3
Follicular study                    3
Hepatitis                           3
Breast lesion                       3
Gallbladder abnormality             

In [16]:
prim_counts = df["primary_label"].value_counts()
KEEP = prim_counts[prim_counts >= 8].index.tolist()
df["primary_label_grouped"] = df["primary_label"].map(
    lambda l: l if l in KEEP else ONTOLOGY.get(l, {}).get("group", "Other")
)
print("Collapsed single-label target used for classification experiments:")
print(df["primary_label_grouped"].value_counts())

Collapsed single-label target used for classification experiments:
primary_label_grouped
Normal study               84
Normal pregnancy           39
Uterine                    15
Early pregnancy failure    14
Ovarian                    13
Hepatobiliary              12
Urinary                    12
Cystitis                   12
Bulky uterus               12
Fatty liver                11
Other                      10
Pelvic                      9
Hepatomegaly                9
Male pelvis                 5
Obstetric                   2
Name: count, dtype: int64


## 11. Save master table

In [17]:
out = df.copy()
out["labels"] = out["labels"].map("|".join)
out["sections_json"] = out["sections"].map(json.dumps)
out = out.drop(columns=["sections"])

cols = ["study_id", "image_path", "has_image", "exam_type",
        "primary_label", "primary_label_grouped", "labels", "n_labels",
        "impression", "advice", "findings", "sections_json", "n_sections",
        "report_clean", "report_norm", "report_raw", "boilerplate_removed"]
out = out[cols]
out.to_csv(ART / "master.csv", index=False)

multihot.to_csv(ART / "labels_multihot.csv")
grouped.to_csv(ART / "labels_grouped.csv")
label_freq.to_csv(ART / "label_frequency.csv", index=False)

print("Saved:")
for p in sorted(ART.iterdir()):
    print(f"  {p.name:26s} {p.stat().st_size/1024:8.1f} KB")

Saved:
  label_audit.csv                27.4 KB
  label_frequency.csv             1.1 KB
  label_tiers.json                1.0 KB
  labels_grouped.csv              5.8 KB
  labels_multihot.csv            18.4 KB
  master.csv                   1274.6 KB
  ontology.json                   6.6 KB


## 12. Dataset card

Paste these numbers straight into Section 3 (Dataset) of the manuscript.

In [18]:
card = {
    "n_studies": int(len(df)),
    "n_with_image": int(df["has_image"].sum()),
    "n_fine_labels": len(ALL_LABELS),
    "n_grouped_labels": len(GROUPS),
    "n_trainable_labels": len(trainable),
    "mean_labels_per_study": round(float(df["n_labels"].mean()), 2),
    "multilabel_studies": int((df["n_labels"] > 1).sum()),
    "mean_report_chars": int(df["report_clean"].str.len().mean()),
    "mean_report_words": int(df["report_clean"].str.split().str.len().mean()),
    "mean_impression_words": int(df["impression"].str.split().str.len().mean()),
    "mean_sections": round(float(df["n_sections"].mean()), 2),
    "exam_types": df["exam_type"].value_counts().to_dict(),
    "unlabelled": int((df["primary_label"] == "Unlabelled").sum()),
}
with open(ART / "dataset_card.json", "w") as f:
    json.dump(card, f, indent=2)
print(json.dumps(card, indent=2))

{
  "n_studies": 259,
  "n_with_image": 259,
  "n_fine_labels": 33,
  "n_grouped_labels": 9,
  "n_trainable_labels": 8,
  "mean_labels_per_study": 1.2,
  "multilabel_studies": 38,
  "mean_report_chars": 972,
  "mean_report_words": 153,
  "mean_impression_words": 7,
  "mean_sections": 8.31,
  "exam_types": {
    "Whole abdomen": 116,
    "Lower abdomen": 56,
    "TVS": 32,
    "Pregnancy profile": 32,
    "Other": 14,
    "Breast": 6,
    "Scrotal": 2,
    "Thyroid/Neck": 1
  },
  "unlabelled": 4
}


---
### Before moving on

1. Open `label_audit.csv` and read every impression against its labels.
   Budget an afternoon. This is the highest-value hour of the whole project.
2. Check that `n_with_image` equals your image count. A silent mismatch here
   poisons every experiment downstream.
3. Extend `ONTOLOGY` for anything printed as `Unlabelled`, then re-run.

Next: **01_eda.ipynb**.